# Supply Chain Demand and Risk Pipeline

This notebook combines the full pipeline used for the FMN supply chain solution:
1. Clean and validate the raw sales data
2. Engineer demand and stock-risk features
3. Build the leakage-safe modeling dataset
4. Train a time-based demand forecasting model
5. Forecast demand and classify SKU risk
6. Save the outputs for the API and frontend

Run the cells in order.

## 0) Exploratory Data Analysis
"""This section helps validate the raw data quality before modeling. It checks the date coverage, category mix, nulls, and sample distribution for demand and inventory."""

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

PIPELINE_DIR = Path.cwd()
RAW_PATH = PIPELINE_DIR / 'raw_data.csv'
CLEANED_PATH = PIPELINE_DIR / 'cleaned_data.csv'
FEATURES_PATH = PIPELINE_DIR / 'features.csv'
MODEL_DATASET_PATH = PIPELINE_DIR / 'model_dataset.csv'
MODEL_PATH = PIPELINE_DIR / 'demand_model.joblib'
MODEL_COLUMNS_PATH = PIPELINE_DIR / 'demand_model_columns.joblib'
RISK_PATH = PIPELINE_DIR / 'risk_flags.csv'
NEW_SKUS = ['SKU-2000', 'SKU-2001', 'SKU-2002']

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'Raw data exists: {RAW_PATH.exists()}')

## 1) Load and clean the raw data

In [ ]:
def load_raw(path=RAW_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Raw data not found: {path}')
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    return df

def normalize_category(df):
    df = df.copy()
    df['category'] = df['category'].str.strip().str.title()
    return df

def drop_exact_duplicates(df):
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates(subset=['sku_id', 'date'], keep='first')
    after = len(df)
    print(f'Dropped {before - after} exact duplicate rows')
    return df

def fill_missing_units_sold(df):
    df = df.copy()
    df = df.sort_values(['sku_id', 'date'])
    df['units_sold_imputed'] = df['units_sold'].isna()
    df['units_sold'] = df.groupby('sku_id')['units_sold'].transform(
        lambda s: s.interpolate(method='linear').round()
    )
    df['units_sold'] = df.groupby('sku_id')['units_sold'].transform(
        lambda s: s.bfill().ffill()
    )
    return df

def reconstruct_missing_closing_stock(df):
    df = df.copy()
    df = df.sort_values(['sku_id', 'date']).reset_index(drop=True)
    df['closing_stock_imputed'] = df['closing_stock'].isna()
    for sku, idx in df.groupby('sku_id').groups.items():
        idx = list(idx)
        for i in idx:
            if pd.isna(df.loc[i, 'closing_stock']):
                pos = idx.index(i)
                if pos == 0:
                    continue
                prev_i = idx[pos - 1]
                prev_stock = df.loc[prev_i, 'closing_stock']
                implied = prev_stock - df.loc[i, 'units_sold'] + df.loc[i, 'units_received']
                df.loc[i, 'closing_stock'] = max(implied, 0)
    return df

def fix_new_sku_lead_time(df):
    df = df.copy()
    df['lead_time_assumed'] = False
    established = df[~df['sku_id'].isin(NEW_SKUS)]
    cat_median_lead = (
        established.groupby(['category', 'sku_id'])['lead_time_days']
        .first()
        .groupby('category')
        .median()
    )
    for sku in NEW_SKUS:
        mask = df['sku_id'] == sku
        if not mask.any():
            continue
        cat = df.loc[mask, 'category'].iloc[0]
        df.loc[mask, 'lead_time_days'] = int(cat_median_lead[cat])
        df.loc[mask, 'lead_time_assumed'] = True
    return df

df = load_raw()
print('Initial shape:', df.shape)
print('Unique categories before normalization:', sorted(df['category'].unique())[:10])

df = normalize_category(df)
df = drop_exact_duplicates(df)
df = fill_missing_units_sold(df)
df = reconstruct_missing_closing_stock(df)
df = fix_new_sku_lead_time(df)

print('Final cleaned shape:', df.shape)
df.head()

In [ ]:
# EDA: inspect the raw sales data
raw_df = load_raw()
print('Columns:', list(raw_df.columns))
print('\nRows x columns:', raw_df.shape)
print('\nMissing values by column:')
print(raw_df.isna().sum().to_string())
print('\nDate range:')
print(raw_df['date'].min(), 'to', raw_df['date'].max())
print('\nCategory distribution:')
print(raw_df['category'].value_counts().head(10).to_string())
print('\nDemand summary:')
print(raw_df['units_sold'].describe().to_string())
print('\nInventory summary:')
print(raw_df['closing_stock'].describe().to_string())

In [ ]:
# Save cleaned CSV
df.to_csv(CLEANED_PATH, index=False)
print(f'Saved cleaned data to {CLEANED_PATH}')

## 2) Engineer demand and risk features

In [ ]:
def load_cleaned(path=CLEANED_PATH):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['sku_id', 'date']).reset_index(drop=True)

def add_rolling_demand_features(df):
    df = df.copy()
    g = df.groupby('sku_id')['units_sold']
    df['avg_sold_7d'] = g.transform(lambda s: s.rolling(7, min_periods=3).mean())
    df['avg_sold_14d'] = g.transform(lambda s: s.rolling(14, min_periods=5).mean())
    df['avg_sold_28d'] = g.transform(lambda s: s.rolling(28, min_periods=7).mean())
    df['std_sold_28d'] = g.transform(lambda s: s.rolling(28, min_periods=7).std())
    return df

def add_volatility_features(df):
    df = df.copy()
    df['cv_sold_28d'] = (df['std_sold_28d'] / df['avg_sold_28d']).replace([np.inf, -np.inf], np.nan)
    return df

def add_trend_feature(df):
    df = df.copy()
    g = df.groupby('sku_id')['units_sold']
    prior_14d = g.transform(lambda s: s.shift(7).rolling(14, min_periods=5).mean())
    df['trend_ratio'] = (df['avg_sold_7d'] / prior_14d).replace([np.inf, -np.inf], np.nan)
    return df

def add_stockout_features(df):
    df = df.copy()
    df['is_stockout_day'] = (df['closing_stock'] == 0).astype(int)
    df['stockout_rate_28d'] = df.groupby('sku_id')['is_stockout_day'].transform(
        lambda s: s.rolling(28, min_periods=5).mean()
    )
    return df

def add_cover_features(df):
    df = df.copy()
    df['days_of_cover'] = df['closing_stock'] / df['avg_sold_14d']
    df['days_of_cover'] = df['days_of_cover'].replace([np.inf, -np.inf], np.nan)
    df['cover_vs_leadtime'] = df['days_of_cover'] / df['lead_time_days']
    return df

clean_df = load_cleaned()
clean_df = add_rolling_demand_features(clean_df)
clean_df = add_volatility_features(clean_df)
clean_df = add_trend_feature(clean_df)
clean_df = add_stockout_features(clean_df)
clean_df = add_cover_features(clean_df)

clean_df.to_csv(FEATURES_PATH, index=False)
print(f'Saved feature table to {FEATURES_PATH}')
clean_df.head()

## 3) Build the leakage-safe modeling dataset

In [ ]:
LAG_FEATURES = [
    'avg_sold_7d', 'avg_sold_14d', 'avg_sold_28d', 'std_sold_28d',
    'cv_sold_28d', 'trend_ratio', 'stockout_rate_28d',
]

def load_features(path=FEATURES_PATH):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['sku_id', 'date']).reset_index(drop=True)

def build_lagged_dataset(df):
    df = df.copy()
    for col in LAG_FEATURES:
        df[f'{col}_lag1'] = df.groupby('sku_id')[col].shift(1)
    df['day_of_week'] = df['date'].dt.dayofweek
    return df

feature_df = load_features()
model_df = build_lagged_dataset(feature_df)

model_cols = [f'{c}_lag1' for c in LAG_FEATURES] + ['day_of_week', 'lead_time_days', 'category', 'sku_id', 'date', 'units_sold']
model_df = model_df[model_cols].copy()
model_df = model_df.dropna(subset=[f'{c}_lag1' for c in LAG_FEATURES])

model_df.to_csv(MODEL_DATASET_PATH, index=False)
print(f'Saved model dataset to {MODEL_DATASET_PATH}')
print(f'Rows after warmup drop: {len(model_df)}')
model_df.head()

## 4) Train the demand forecasting model

In [ ]:
HOLDOUT_DAYS = 14
NUMERIC_FEATURES = [
    'avg_sold_7d_lag1', 'avg_sold_14d_lag1', 'avg_sold_28d_lag1',
    'std_sold_28d_lag1', 'cv_sold_28d_lag1', 'trend_ratio_lag1',
    'stockout_rate_28d_lag1', 'day_of_week', 'lead_time_days',
]
CATEGORICAL_FEATURES = ['category']

def time_based_split(df, holdout_days=HOLDOUT_DAYS):
    cutoff_per_sku = df.groupby('sku_id')['date'].transform(
        lambda d: d.max() - pd.Timedelta(days=holdout_days)
    )
    train = df[df['date'] <= cutoff_per_sku].copy()
    test = df[df['date'] > cutoff_per_sku].copy()
    return train, test

def build_design_matrix(df, category_columns=None):
    X = pd.get_dummies(df[CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES)
    X = pd.concat([df[NUMERIC_FEATURES].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    if category_columns is not None:
        X = X.reindex(columns=category_columns, fill_value=0)
    return X

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])).mean() * 100

df_model = pd.read_csv(MODEL_DATASET_PATH)
df_model['date'] = pd.to_datetime(df_model['date'])
train, test = time_based_split(df_model)

X_train = build_design_matrix(train)
y_train = train['units_sold']
X_test = build_design_matrix(test, category_columns=X_train.columns)
y_test = test['units_sold']

baseline_pred = test['avg_sold_14d_lag1']
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mape = mean_absolute_percentage_error(y_test, baseline_pred)

model = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, model_pred)
model_mape = mean_absolute_percentage_error(y_test, model_pred)

print('=== Backtest results ===')
print(f'Naive MAE: {baseline_mae:.2f}')
print(f'Naive MAPE: {baseline_mape:.1f}%')
print(f'Random Forest MAE: {model_mae:.2f}')
print(f'Random Forest MAPE: {model_mape:.1f}%')
print(f'MAE improvement over naive baseline: {(1 - model_mae / baseline_mae) * 100:.1f}%')

joblib.dump(model, MODEL_PATH)
joblib.dump(list(X_train.columns), MODEL_COLUMNS_PATH)
print(f'Saved model to {MODEL_PATH}')
print(f'Saved model columns to {MODEL_COLUMNS_PATH}')

## 5) Generate demand forecasts and risk flags

In [ ]:
def load_features_for_prediction(path=FEATURES_PATH):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['sku_id', 'date']).reset_index(drop=True)

def get_latest_snapshot(df):
    latest = df.sort_values('date').groupby('sku_id').tail(1).copy()
    next_date = df['date'].max() + pd.Timedelta(days=1)
    latest['day_of_week'] = next_date.dayofweek
    return latest, next_date

def predict_established(latest_established, model, model_columns):
    df = latest_established.copy()
    rename_map = {
        c: f'{c}_lag1' for c in [
            'avg_sold_7d', 'avg_sold_14d', 'avg_sold_28d', 'std_sold_28d',
            'cv_sold_28d', 'trend_ratio', 'stockout_rate_28d'
        ]
    }
    df = df.rename(columns=rename_map)
    numeric_cols = list(rename_map.values()) + ['day_of_week', 'lead_time_days']
    X = pd.get_dummies(df[['category']].copy(), columns=['category'])
    X = pd.concat([df[numeric_cols].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    missing = set(model_columns) - set(X.columns)
    if missing:
        raise ValueError(f'Column mismatch: {missing}')
    X = X.reindex(columns=model_columns, fill_value=0)
    return model.predict(X)

def predict_new_skus(latest_new, df_all):
    established = df_all[~df_all['sku_id'].isin(NEW_SKUS)]
    recent_28d = established.sort_values('date').groupby('sku_id').tail(28)
    cat_avg = recent_28d.groupby('category')['units_sold'].mean()
    return latest_new['category'].map(cat_avg).values

def classify_risk(row):
    if row['projected_demand_leadtime'] == 0:
        return 'HEALTHY'
    ratio = row['closing_stock'] / row['projected_demand_leadtime']
    if ratio < 1.0:
        return 'STOCKOUT_RISK'
    elif ratio > 2.0:
        return 'OVERSTOCK'
    return 'HEALTHY'

feature_df = load_features_for_prediction()
model = joblib.load(MODEL_PATH)
model_columns = joblib.load(MODEL_COLUMNS_PATH)

latest, next_date = get_latest_snapshot(feature_df)
print(f'Forecasting for: {next_date.date()}')

is_new = latest['sku_id'].isin(NEW_SKUS)
latest_established = latest[~is_new].copy()
latest_new = latest[is_new].copy()

latest_established['predicted_next_day_demand'] = predict_established(latest_established, model, model_columns)
latest_established['confidence'] = 'normal'

latest_new['predicted_next_day_demand'] = predict_new_skus(latest_new, feature_df)
latest_new['confidence'] = 'low (new SKU, category-average fallback)'

result = pd.concat([latest_established, latest_new], ignore_index=True)
result['projected_demand_leadtime'] = (result['predicted_next_day_demand'] * result['lead_time_days']).round(1)
result['risk_flag'] = result.apply(classify_risk, axis=1)
result['cover_ratio'] = (result['closing_stock'] / result['projected_demand_leadtime']).round(2)

out_cols = [
    'sku_id', 'category', 'closing_stock', 'lead_time_days',
    'predicted_next_day_demand', 'projected_demand_leadtime',
    'cover_ratio', 'risk_flag', 'confidence',
]
result = result[out_cols].sort_values('cover_ratio')

result.to_csv(RISK_PATH, index=False)
print('Risk flags saved to', RISK_PATH)
result.head(15)

## 6) Final summary and outputs

The notebook writes the following artifacts for the rest of the app:

In [ ]:
print(f'Cleaned data: {CLEANED_PATH.exists()} -> {CLEANED_PATH}')
print(f'Features data: {FEATURES_PATH.exists()} -> {FEATURES_PATH}')
print(f'Model dataset: {MODEL_DATASET_PATH.exists()} -> {MODEL_DATASET_PATH}')
print(f'Model file: {MODEL_PATH.exists()} -> {MODEL_PATH}')
print(f'Risk flags: {RISK_PATH.exists()} -> {RISK_PATH}')
print('\nRisk summary:')
print(result['risk_flag'].value_counts().to_string())